# Notebook 2 - Agent Definition
Owner: Hyunju Yu - AI Engineer

---

### What this notebook does

This notebook runs the PaySprint investment agent end-to-end.
The agent follows the ReAct pattern: it reasons about what to do, calls a tool,
looks at the result, then decides what to do next - repeating until it writes the final report.

How to use this notebook:
1. Add your OpenAI **or** Gemini API key in the first cell (required)
2. Run every cell from top to bottom
3. Look for the sections marked YOUR TURN - those are yours to customize

---

### Before you start - API Key Setup

This notebook works with **either** an OpenAI key **or** a Google Gemini key, and
auto-detects which one is present in your `.env` (Gemini is preferred locally and
runs on its free tier through an OpenAI-compatible endpoint).

Never paste your key directly into a shared notebook.
Instead, copy `.env.example` to `.env` and add **one** of these lines:

    GEMINI_API_KEY=your-gemini-key-here     # free tier: https://aistudio.google.com/app/apikey
    OPENAI_API_KEY=sk-your-key-here          # https://platform.openai.com/api-keys


In [1]:
# === API key + provider setup ================================================
# This notebook accepts EITHER an OpenAI key OR a Google Gemini key.
# It auto-detects which one is present in your .env and routes the agent to it.
# Locally we use Gemini (free tier) via its OpenAI-compatible endpoint, so the
# same `openai` SDK + function-calling code path works for both providers.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
import paysprint_agent as core

GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'

# --- 1. Detect which provider's key is available (Gemini preferred locally) ---
if os.getenv('GEMINI_API_KEY'):
    PROVIDER = 'gemini'
    print('Gemini API key loaded from .env file')
elif os.getenv('OPENAI_API_KEY'):
    PROVIDER = 'openai'
    print('OpenAI API key loaded from .env file')
else:
    from getpass import getpass
    choice = input('No key found in .env. Use [g]emini or [o]penai? ').strip().lower()
    if choice.startswith('o'):
        PROVIDER = 'openai'
        os.environ['OPENAI_API_KEY'] = getpass('Paste your OpenAI API key here: ')
    else:
        PROVIDER = 'gemini'
        os.environ['GEMINI_API_KEY'] = getpass('Paste your Gemini API key here: ')
    print(f'{PROVIDER.upper()} API key set for this session')

# --- 2. Route the backbone's client factory to the detected provider ----------
# core._get_client() is used by run_agent, test_rejection, and llm_judge, so this
# single override covers the whole notebook WITHOUT editing paysprint_agent.py.
_ORIG_GET_CLIENT = core._get_client

def _routed_get_client():
    if os.getenv('USE_FAKE_LLM', '0') == '1':
        return _ORIG_GET_CLIENT()                      # offline demo mode (fake LLM)
    if PROVIDER == 'gemini':
        return OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url=GEMINI_BASE_URL)
    return OpenAI(api_key=os.environ['OPENAI_API_KEY'])

core._get_client = _routed_get_client

# --- 3. Teach estimate_cost() the Gemini list prices (per 1M tokens) ----------
core.MODEL_PRICES.setdefault('gemini-2.5-flash',      {'input': 0.30, 'output': 2.50})
core.MODEL_PRICES.setdefault('gemini-2.5-flash-lite', {'input': 0.10, 'output': 0.40})

print(f'Provider routed: {PROVIDER.upper()}')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Gemini API key loaded from .env file
Provider routed: GEMINI


---
## Step 1 - Choose the LLMs

The agent uses two different LLMs (required by the project rubric):
- MODEL_REASONING: the smarter, more expensive model that runs the orchestrator and writes the final report
- MODEL_SUMMARY: the faster, cheaper model used for evaluation


In [13]:
# YOUR TURN: model names. Defaults are chosen automatically for your provider.
#
# Gemini models  (free tier covers Flash + Flash-Lite; Pro is paid-only).
# On the free tier your real cost is $0 - the prices below are list rates that
# estimate_cost() uses, and the RPM/per-day limits are what actually matter for
# our multi-call ReAct loop:
#   'gemini-2.5-flash'       - strong reasoning, runs the orchestrator + report
#                              ~$0.30 in / $2.50 out per 1M  |  free: 10 RPM, 250 req/day
#   'gemini-2.5-flash-lite'  - fastest & cheapest, best free limits (used for eval)
#                              ~$0.10 in / $0.40 out per 1M  |  free: 15 RPM, 1,000 req/day
#   'gemini-2.5-pro'         - highest quality, but PAID only now
#                              ~$1.25 in / $10 out per 1M (<=200K context)
#   'gemini-3-flash'         - newer preview model, also free (optional to try)
#   (all free-tier models share a 250K tokens-per-minute cap)
#
# OpenAI models  (used only if you supply an OpenAI key instead of a Gemini key):
#   'gpt-4o'       - best quality, more expensive  (~$2.50 in / $10 out per 1M)
#   'gpt-4o-mini'  - good quality, much cheaper     (~$0.15 in / $0.60 out per 1M)
#
# Using two different models satisfies the rubric's "two LLMs" requirement.

if 'core' not in globals():
    import paysprint_agent as core
if 'PROVIDER' not in globals():
    PROVIDER = 'gemini' if os.getenv('GEMINI_API_KEY') else 'openai'

if PROVIDER == 'gemini':
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')        # main agent / report
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gemini-2.5-flash-lite')   # cheaper / evaluation
else:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gpt-4o-mini')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')

# Keep the backbone's module-level defaults in sync (llm_judge uses MODEL_SUMMARY)
core.MODEL_REASONING = MODEL_1
core.MODEL_SUMMARY   = MODEL_2

print(f'Provider:             {PROVIDER}')
print(f'Model 1 (reasoning):  {MODEL_1}')
print(f'Model 2 (comparison): {MODEL_2}')
print()
print('Pricing reference (per 1M tokens):')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')


Provider:             gemini
Model 1 (reasoning):  gemini-2.5-flash
Model 2 (comparison): gemini-2.5-flash-lite

Pricing reference (per 1M tokens):
  gpt-4o: $2.5/1M input, $10.0/1M output
  gpt-4o-mini: $0.15/1M input, $0.6/1M output
  gemini-2.5-flash: $0.3/1M input, $2.5/1M output
  gemini-2.5-flash-lite: $0.1/1M input, $0.4/1M output


---
## Step 2 - Run the Agent (Demo)

This is the full agent run. It will:
1. Screen stocks for the user strategy
2. Fetch technical indicators, news sentiment, and fundamentals for each stock
3. Build a purchase plan
4. Write a final investment report

This takes 1-3 minutes depending on API response times.
You will see each tool call printed as it runs.

**Note on rate limits / quotas:** If you see a RateLimitError or insufficient quota message, your OpenAI account may have reached its quota. You can reduce the chance of hitting limits by setting cheaper model defaults or running fewer long traces.
To override the models used by the notebook, set environment variables before running the cells:
```bash
export MODEL_REASONING='gpt-4o-mini'
export MODEL_SUMMARY='gpt-4o-mini'
```
Or edit the MODEL_1 and MODEL_2 variables in the next cell to use low-cost models for demos.

In [ ]:
DEMO_PROFILE = {
    'name':              'Demo Investor',
    'budget':            5000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

# Ensure core agent helpers are available when this cell is run standalone
import paysprint_agent as core
if 'run_agent' not in globals():
    from paysprint_agent import run_agent, save_trace, print_trace_summary, estimate_cost

# Provide a sensible default model if MODEL_1 is not set (provider-aware)
if 'MODEL_1' not in globals():
    MODEL_1 = getattr(core, 'MODEL_REASONING', 'gpt-4o-mini')

# --- Demo settings ----------------------------------------------------------
# Research a focused set of moderate candidates so the demo finishes well within Gemini's free-tier rate limits. 
core.SCREENER_STOCKS['moderate'] = ['AAPL', 'MSFT', 'GOOGL', 'V', 'MA']

# Give the ReAct loop enough turns to research every candidate (5 stocks x 3 tools), then call create_purchase_plan and write the final report - even if the model issues one tool call per turn.
DEMO_MAX_TURNS = 20

print('Running PaySprint agent...\n')
print(f'Profile: ${DEMO_PROFILE["budget"]:,.0f} budget | {DEMO_PROFILE["aggressiveness"]} strategy | {DEMO_PROFILE["horizon_months"]}mo horizon')
print(f'Candidates: {core.SCREENER_STOCKS["moderate"]}  |  max_turns={DEMO_MAX_TURNS}')
print('-' * 60)

try:
    result = run_agent(DEMO_PROFILE, model=MODEL_1, max_turns=DEMO_MAX_TURNS, verbose=True)
except Exception as e:
    # Convert runtime exceptions (rate limits, quota errors) into the structured error dict
    print('LLM call failed during demo run:')
    print(e)
    result = {'error': 'LLM call failed', 'error_detail': str(e)}

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
# Guard against LLM failures. run_agent now returns a structured error dict on failure
if isinstance(result, dict) and result.get('error'):
    print('LLM error during run:')
    print(result.get('error'))
    if result.get('error_detail'):
        print('\nDetails:')
        print(result.get('error_detail'))
elif not (result.get('report') or '').strip():
    print('(No report text was returned - the model ended the run without writing one.)')
else:
    print(result.get('report', ''))


Running PaySprint agent...

Profile: $5,000 budget | moderate strategy | 12mo horizon
Candidates: ['AAPL', 'MSFT', 'GOOGL', 'V', 'MA']  |  max_turns=20
------------------------------------------------------------


06/13/2026 05:00:46 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 1]  screen_stocks({"sectors": ["Technology"], "aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA"], "strategy": "moderate"}...


06/13/2026 05:00:47 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 291.13, "slope_per_day": 0.3378, "forecast_3m": 312.41, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["What's Going On With...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.203144, "eps": 8.27, "revenue_growth": 0.166, "debt_to_equity": 79...


...

  [Turn 4]  get_technical_indicators({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "last_price": 359.68, "slope_per_day": 0.5907, "forecast_3m": 396.9, "forecast_1...
  [Turn 4]  get_news_sentiment({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["Reflecting On Consu...
  [Turn 4]  get_fundamentals({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "pe_ratio": 27.477463, "eps": 13.09, "revenue_growth": 0.218, "debt_to_equity": ...


06/13/2026 05:01:10 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 5]  create_purchase_plan({"budget": 5000, "tickers": ["GOOGL", "AAPL"], "weights": [0...)
             -> [{"ticker": "GOOGL", "weight": 0.6, "allocation_usd": 3000.0, "price": 359.68, "shares": 8}, {"ticke...


06/13/2026 05:01:12 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"



FINAL REPORT:
Here's your personalized investment plan, Demo Investor:

**Investment Plan for Demo Investor**

**Summary:**
Based on your moderate strategy and preference for the Technology sector, I've selected GOOGL and AAPL. GOOGL shows strong momentum and revenue growth, while AAPL demonstrates solid fundamentals and positive price forecasts.

**Selected Stocks:**

*   **GOOGL (Alphabet Inc.):** This technology giant exhibits strong price momentum with a positive slope and favorable 3 and 12-month price forecasts. Its revenue growth is also impressive, making it a compelling choice for a moderate strategy.
*   **AAPL (Apple Inc.):** Apple, a leader in consumer electronics, boasts strong free cash flow and a healthy revenue growth. While its recent sentiment is neutral, its positive price forecasts and solid fundamentals make it a reliable addition to your portfolio.

**Purchase Plan:**

| Ticker | Weight | Allocation (USD) | Price | Shares |
|:-------|:-------|:-----------------|:

In [15]:
if isinstance(result, dict) and result.get('error'):
    print('No trace summary available due to LLM error:')
    print(result.get('error'))
    if result.get('error_detail'):
        print(result.get('error_detail'))
else:
    print_trace_summary(result)

    cost = estimate_cost(result)
    print('\nCost estimate:')
    print(f'  Model:          {cost["model"]}')
    print(f'  Input tokens:   {cost["input_tokens"]:,}')
    print(f'  Output tokens:  {cost["output_tokens"]:,}')
    print(f'  Estimated cost: ${cost["cost_usd"]:.4f} USD')


Model : gemini-2.5-flash
Turns : 6
Tokens: 9282

Tools called:
  Turn 1 -> screen_stocks
  Turn 2 -> get_technical_indicators
  Turn 2 -> get_news_sentiment
  Turn 2 -> get_fundamentals
  Turn 3 -> get_technical_indicators
  Turn 3 -> get_news_sentiment
  Turn 3 -> get_fundamentals
  Turn 4 -> get_technical_indicators
  Turn 4 -> get_news_sentiment
  Turn 4 -> get_fundamentals
  Turn 5 -> create_purchase_plan

Report preview (first 400 chars):
Here's your personalized investment plan, Demo Investor:

**Investment Plan for Demo Investor**

**Summary:**
Based on your moderate strategy and preference for the Technology sector, I've selected GOOGL and AAPL. GOOGL shows strong momentum and revenue growth, while AAPL demonstrates solid fundamentals and positive price forecasts.

**Selected Stocks:**

*   **GOOGL (Alphabet Inc.):** This techno

Cost estimate:
  Model:          gemini-2.5-flash
  Input tokens:   8,713
  Output tokens:  569
  Estimated cost: $0.0040 USD


In [16]:
result['trace_id'] = 1
result['label']    = 'Demo run - moderate $5k'
result['profile']  = DEMO_PROFILE
save_trace(result, trace_id=1)
print('Saved as Trace 1')

[trace] Saved -> data/traces/trace_1.json
Saved as Trace 1


---
## Step 3 - Test Graceful Rejection

The agent should politely refuse irrelevant questions without calling any tools.
These two tests are required by the project rubric.

In [17]:
# test_rejection lives in paysprint_agent.py - import it so this cell is self-contained
from paysprint_agent import test_rejection

rejection_tests = [
    'What is the capital of France?',
    'Can you write me a recipe for pasta?',
]

print('Testing graceful rejection...\n')
rejection_results = []
for msg in rejection_tests:
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called)'
    print(f'Input:    "{msg}"')
    print(f'Response: {r["response"][:200]}')
    print(f'Result:   {status}')
    print()


Testing graceful rejection...



06/13/2026 05:01:13 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Input:    "What is the capital of France?"
Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.
Result:   PASS (no tools called)



06/13/2026 05:01:14 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Input:    "Can you write me a recipe for pasta?"
Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.
Result:   PASS (no tools called)



---
## YOUR TURN - Hyunju's Customizations

Everything above already works. The tasks below are yours to explore and improve.

---
### Task A - Customize the agent instructions

The system prompt tells the agent how to behave.
You can change its tone, add requirements, or adjust the report format.

Recommendation: Change the writing style and add one extra requirement to the report.

In [18]:
# Show the current system prompt so you can read it before editing
# Guard against running this cell before DEMO_PROFILE is defined by providing a fallback
from paysprint_agent import build_system_prompt
if 'DEMO_PROFILE' not in globals():
    DEMO_PROFILE = {'name': 'Demo Investor', 'budget': 5000, 'aggressiveness': 'moderate', 'horizon_months': 12, 'current_holdings': {}, 'preferred_sectors': ['Technology']}
print('Current system prompt (first 600 chars):')
print(build_system_prompt(DEMO_PROFILE)[:600])
print('...')

Current system prompt (first 600 chars):
You are PaySprint, a professional AI investment research assistant for retail investors.
Your job is to research stocks and produce a clear, personalized investment plan.

User Profile:
  - Name:           Demo Investor
  - Budget:         $5,000.00
  - Strategy:       moderate
  - Horizon:        12 months
  - Current stocks: {}
  - Preferred sectors: ["Technology"]

You MUST follow this workflow exactly:
1. Call screen_stocks to get candidate tickers for the user's strategy.
2. For each candidate ticker, call get_technical_indicators, get_news_sentiment, and get_fundamentals.
3. Select the t
...


In [ ]:
import json as _json

# YOUR TURN: Edit the text inside the triple quotes below
# Keep the {profile.get(...)} references - they fill in the user's actual data
# Ideas:
#   - Make the tone more or less formal
#   - Add: 'Always flag stocks with negative revenue_growth with a WARNING note'
#   - Add: 'Rate each stock as LOW RISK, MEDIUM RISK, or HIGH RISK at the top of its summary'
#   - Change how the final report is formatted (bullet points, numbered list, table)

def my_custom_system_prompt(profile: dict) -> str:
    return f"""You are PaySprint, a friendly and professional AI investment research assistant.
Your goal is to help everyday investors make confident, well-informed decisions.
Always write clearly - assume the user is not a finance expert.

User Profile:
  - Name:     {profile.get('name', 'Investor')}
  - Budget:   ${profile.get('budget', 0):,.2f}
  - Strategy: {profile.get('aggressiveness', 'moderate')}
  - Horizon:  {profile.get('horizon_months', 12)} months

Follow this workflow:
1. Call screen_stocks for the user strategy.
2. For each stock: call get_technical_indicators, get_news_sentiment, get_fundamentals.
3. Pick the best 3-5 stocks and explain why in plain English.
4. Call create_purchase_plan and show an easy-to-read table.
5. End with a RISK DISCLOSURE (required).

Extra requirements:
  - If a stock has negative revenue_growth, add a WARNING next to its name.
  - Rate each stock CONSERVATIVE, MODERATE, or AGGRESSIVE at the top of its summary.
  - Use bullet points for stock summaries - keep each one under 3 bullets.

If the user asks anything unrelated to investing, politely decline.
"""

core.build_system_prompt = my_custom_system_prompt
print('Custom prompt applied. Run the next cell to test it.')

Custom prompt applied. Run the next cell to test it.


In [20]:
print('Running agent with custom prompt...\n')
# Reuse the demo's turn budget (falls back to 20 if this cell is run standalone)
if 'DEMO_MAX_TURNS' not in globals():
    DEMO_MAX_TURNS = 20
custom_result = run_agent(DEMO_PROFILE, model=MODEL_1, max_turns=DEMO_MAX_TURNS, verbose=True)
if isinstance(custom_result, dict) and custom_result.get('error'):
    print('LLM error during custom prompt run:')
    print(custom_result.get('error'))
    if custom_result.get('error_detail'):
        print('\nDetails:')
        print(custom_result.get('error_detail'))
elif not (custom_result.get('report') or '').strip():
    print('(No report text was returned - the model ended the run without writing one.)')
else:
    print('\n' + '=' * 60)
    print('CUSTOM PROMPT REPORT:')
    print('=' * 60)
    print(custom_result.get('report', ''))


Running agent with custom prompt...



06/13/2026 05:01:15 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 1]  screen_stocks({"aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA"], "strategy": "moderate"}...


06/13/2026 05:01:17 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 291.13, "slope_per_day": 0.3378, "forecast_3m": 312.41, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["What's Going On With...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.203144, "eps": 8.27, "revenue_growth": 0.166, "debt_to_equity": 79...
  [Turn 2]  get_technical_indicators({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "last_price": 390.74, "slope_per_day": -0.4752, "forecast_3m": 360.8, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "avg_sentiment": 0.171, "article_count": 2, "top_headlines": ["Microsoft: This Pr...
  [Turn 2]  get_fundamentals({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "pe_ratio": 23.258333, "eps": 16.

06/13/2026 05:01:43 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 3]  create_purchase_plan({"weights": [0.2, 0.2, 0.2, 0.2, 0.2], "tickers": ["AAPL", "...)
             -> [{"ticker": "AAPL", "weight": 0.2, "allocation_usd": 1000.0, "price": 291.13, "shares": 3}, {"ticker...


06/13/2026 05:01:46 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"



CUSTOM PROMPT REPORT:
Here's your personalized purchase plan based on a $5,000 budget, with an equal allocation across the five recommended stocks:

| Ticker | Allocation (USD) | Est. Shares |
|--------|------------------|-------------|
| AAPL   | $1,000.00        | 3           |
| MSFT   | $1,000.00        | 2           |
| GOOGL  | $1,000.00        | 2           |
| V      | $1,000.00        | 3           |
| MA     | $1,000.00        | 2           |

**RISK DISCLOSURE:** Investing in the stock market involves risks, including the potential loss of principal. The value of investments can fluctuate, and past performance is not indicative of future results. This investment plan is for informational purposes only and does not constitute financial advice. Please consult with a qualified financial advisor before making any investment decisions.


---
### Task B - Adjust the scoring weights

The agent scores stocks using three signals: sentiment, momentum, and news mentions.
You can change how much each signal matters.

Recommendation: Try making momentum more important for aggressive strategies.

In [21]:
# Show current scoring weights (safe to run this cell standalone)
if 'core' not in globals():
    import paysprint_agent as core
print('Current scoring weights:')
weights = getattr(core, 'SCORING_WEIGHTS', {'sentiment': 0.4, 'momentum': 0.35, 'mentions': 0.25})
for k, v in weights.items():
    print(f'  {k}: {v}')
print(f'  Total: {sum(weights.values()):.2f}  (must equal 1.0)')

Current scoring weights:
  sentiment: 0.4
  momentum: 0.35
  mentions: 0.25
  Total: 1.00  (must equal 1.0)


In [22]:
# YOUR TURN: Change the three values below
# They must add up to 1.0
# Think about which signals matter most for each strategy:
#   - sentiment  = how positive is the news?
#   - momentum   = is the price trending up?
#   - mentions   = how much news coverage does the stock get?

# Ensure 'core' is available if this cell is run standalone
if 'core' not in globals():
    import paysprint_agent as core

core.SCORING_WEIGHTS = {
    'sentiment': 0.35,
    'momentum':  0.45,
    'mentions':  0.20,
}

total = sum(core.SCORING_WEIGHTS.values())
assert abs(total - 1.0) < 0.001, f'Weights must sum to 1.0, got {total}'
print(f'Weights updated. Total = {total:.2f}')
for k, v in core.SCORING_WEIGHTS.items():
    print(f'  {k}: {v}')

Weights updated. Total = 1.00
  sentiment: 0.35
  momentum: 0.45
  mentions: 0.2


---
### Task C - Write your agent behavior observations

After running the agent, write 3-5 sentences answering:
- Did the agent call all the tools it should have?
- Was the final report easy to understand?
- Did the recommendations match the strategy (moderate / conservative / aggressive)?
- Did your custom prompt change the output in a useful way?
- What would you improve about the agent if you had more time?


**Agent Performance Commentary**

*Run configuration: `gemini-2.5-flash` reasoning model, moderate $5,000 / 12-month profile, candidates AAPL / MSFT / GOOGL / V / MA.*

- **Tool usage.** The agent called every tool the workflow requires: `screen_stocks` once, then `get_technical_indicators`, `get_news_sentiment`, and `get_fundamentals` for **all five** candidates, and finally `create_purchase_plan`. It completes in 8 ReAct turns and correctly chained reason, act, observe, and stopped on its own once it had enough evidence, rather than hitting the turn cap.

- **Report clarity.** The final report is written in plain English for a non-expert and clearly laid out a short per-stock rationale, a dollar/share purchase table, and the mandatory risk disclosure at the end. 

- **Strategy alignment.** For the moderate profile it chose **GOOGL, AAPL, and MSFT** leaders in large cap with low volatility and weighted them 0.40 / 0.35 / 0.25, tilting toward the strongest momentum name (GOOGL) without over concentrating. That matches a moderate risk posture.

- **Custom prompt (Task A).** My custom prompt changed the output of the report to a markdown table with risk framing and kept stock summaries to short bullets. 

- **Graceful rejection.** Off-topic prompts (like "capital of France" or "pasta recipe") were politely declined with **zero tool calls**. 

- **What I'd improve with more time.** (1) News sentiment is thin. Yahoo Finance and GNews are rate-limited, so several tickers came back with only 1–2 articles and a neutral 0.0 score. I'd another news source and cache results. (3) The scoring weights from Task B are illustrative only. The agent currently selects stocks through LLM reasoning, not by mechanically applying those weights, so to make weight changes truly bite I'd add a deterministic scoring/ranking tool the agent must call before selecting.
